# Traffic-light YOLO training

Interactive version of `train_bulb_color_detector_experiment.py` - use this when you want to tweak hyperparameters/augmentation and re-run a cell instead of re-launching a whole CLI training run each time.

Run `split_yolo_dataset.py` at least once first (or run the cell below) so `../../datasets/traffic_light/train.txt`/`val.txt` exist.

In [ ]:
import os
from pathlib import Path

from ultralytics import YOLO

TRAINING_DIR = Path.cwd()
DATASET_DIR = (TRAINING_DIR / ".." / "dataset").resolve()
DATA_YAML = DATASET_DIR / "data.yaml"

assert DATA_YAML.exists(), f"data.yaml not found at {DATA_YAML}"
print("dataset dir:", DATASET_DIR)
print("data.yaml:", DATA_YAML)

## 1. Train/val split

Re-run this cell any time after collecting more frames - it reshuffles from scratch (fixed seed) and overwrites `train.txt`/`val.txt`. Adjust `VAL_FRACTION` here if you want a bigger/smaller validation slice.

In [ ]:
import random

VAL_FRACTION = 0.15
SEED = 0

images_dir = DATASET_DIR / "images"
basenames = sorted(f.name for f in images_dir.glob("*.jpg"))
random.Random(SEED).shuffle(basenames)
n_val = max(1, int(len(basenames) * VAL_FRACTION))
val_set = set(basenames[:n_val])
train_set = [b for b in basenames if b not in val_set]

(DATASET_DIR / "train.txt").write_text("\n".join(f"images/{b}" for b in train_set) + "\n")
(DATASET_DIR / "val.txt").write_text("\n".join(f"images/{b}" for b in sorted(val_set)) + "\n")

print(f"{len(basenames)} total | train {len(train_set)} | val {len(val_set)}")

## 2. Hyperparameters + augmentation

Worth leaning on augmentation here specifically because the collected dataset is small and repetitive (long stretches of near-identical frames from waiting at one red light), so the more variety augmentation can synthesize per epoch, the better. Defaults below are the tuned starting point, not fixed - change anything and re-run:

- `mosaic=1.0`: keep at full strength (default) - stitches 4 images into one training sample, the single biggest lever for squeezing more effective diversity out of a small/redundant dataset.
- `hsv_h=0.01`: kept **low, at/below** ultralytics' default (0.015) - deliberately, not increased. Hue *is* the class signal here (red vs. yellow vs. green); shifting it aggressively can flip what a lit bulb visually looks like while the label stays the same, teaching the model the wrong color for the class. Only `s`/`v` are pushed up.
- `hsv_s=0.8`, `hsv_v=0.5`: pushed above defaults - saturation/brightness jitter is safe (doesn't change hue) and helps generalize past this sim's exact lighting/exposure.
- `degrees`, `shear`, `perspective`: kept small/off - the camera is always roughly upright and forward-facing (dash-cam-like), heavy rotation would create geometry the real deployment never sees.
- `scale=0.3`: mild only - lights are already tiny (single-digit to ~20px), aggressively shrinking further can make them vanish below the label's effective size.
- `flipud=0.0`: off - a street scene upside down is never physically real.
- `fliplr=0.5`: on (default) - a mirrored street/intersection is still a valid scene, cheap free diversity.
- `mixup=0.0`: off by default - blends two whole images together, tends to help more on larger datasets; try turning it on (e.g. 0.1) if you see overfitting in the loss curves.

In [ ]:
from train import _write_absolute_data_yaml

ABS_DATA_YAML = _write_absolute_data_yaml(str(DATASET_DIR))

train_args = dict(
    data=ABS_DATA_YAML,
    model="yolov8n.pt",   # or yolov8s.pt for more capacity if training is fast enough
    epochs=100,
    imgsz=640,
    batch=16,
    workers=0,            # more reliable on Windows/Jupyter

    # augmentation - see markdown above for the reasoning behind each
    mosaic=1.0,
    hsv_h=0.01,
    hsv_s=0.8,
    hsv_v=0.5,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    scale=0.3,
    translate=0.1,
    flipud=0.0,
    fliplr=0.5,
    mixup=0.0,

    project=str(TRAINING_DIR / "runs" / "detect"),
    name="trafficlight_jupyter",
)
train_args


## 3. Train

`name="trafficlight"` reused across runs will auto-increment (`trafficlight2`, `trafficlight3`, ...) so old runs aren't overwritten - convenient for comparing hyperparameter changes side by side under `../output/runs/detect/`.

In [ ]:
model = YOLO(train_args.pop("model"))
results = model.train(**train_args)
run_dir = Path(model.trainer.save_dir)
print("run dir:", run_dir)
print("best weights:", run_dir / "weights" / "best.pt")

## 4. Inspect results

Ultralytics writes these directly into `run_dir` - display the ones worth eyeballing before deciding whether to re-tune and re-run cell 3.

In [ ]:
from IPython.display import Image, display


for name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg", "PR_curve.png"]:
    path = run_dir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))
    else:
        print(f"(missing: {name})")

In [ ]:
!python preview_yolo_predictions.py --model ../output/runs/detect/trafficlight_color_experiment/weights/best.pt --count 80

In [ ]:
model.predict(
    source="../dataset/images/wp025_00223_1784677289.456.jpg",
    imgsz=640,
    conf=0.25,
    save=True,
    project="runs/predict",
    name="one_image",
    line_width=1,
    show_labels=False,
    show_conf=False,
)

In [ ]:
!python preview_yolo_predictions.py --model ../output/runs/detect/trafficlight_color_experiment/weights/best.pt --count 30 --no-labels